# 03 — Embedding Model Experiments

Compare embedding models on:
- Vector dimensionality
- Cosine similarity between related vs unrelated pairs
- Encoding latency

**Models to test (all via Ollama unless noted):**
- `nomic-embed-text` — current baseline
- `mxbai-embed-large`
- `snowflake-arctic-embed`
- `all-minilm` (small, fast)

> Pull a model with: `ollama pull <model-name>`

In [ ]:
import sys, os, time
from pathlib import Path

repo_root = str(Path(os.getcwd()).parents[1] / "RAG_Ai_Assistant_Uni")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_ollama import OllamaEmbeddings

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 1. Define Test Pairs

High-similarity pairs should score closer to 1.0; unrelated pairs closer to 0.

In [ ]:
TEST_PAIRS = [
    # (label, text_a, text_b, expected_relation)
    ("related-1",
     "Alphawave provides AI consulting services.",
     "We offer artificial intelligence strategy and advisory.",
     "high"),
    ("related-2",
     "What are the opening hours of the sports center?",
     "The sports center is open from 7am to 10pm.",
     "high"),
    ("unrelated-1",
     "Alphawave provides AI consulting services.",
     "The weather today is sunny with a chance of rain.",
     "low"),
    ("unrelated-2",
     "Digital transformation strategy for enterprises.",
     "The recipe calls for two cups of flour.",
     "low"),
    ("edge-1",
     "Machine learning for business automation.",
     "Automating workflows using AI models.",
     "medium"),
]

print(f"{len(TEST_PAIRS)} test pairs defined.")

## 2. Baseline: nomic-embed-text

In [ ]:
from app.embeddings import generate_embedding, EMBED_MODEL

print(f"Current model: {EMBED_MODEL}")

emb = generate_embedding("test")
print(f"Dimension: {len(emb)}")

for label, a, b, expected in TEST_PAIRS:
    sim = cosine_sim(generate_embedding(a), generate_embedding(b))
    print(f"  [{expected:6}] {label}: {sim:.4f}")

## 3. Multi-Model Comparison

Add/remove models from `MODELS` as needed. Each model must be pulled in Ollama first.

In [ ]:
MODELS = [
    "nomic-embed-text",
    # "mxbai-embed-large",
    # "snowflake-arctic-embed",
    # "all-minilm",
]

results = []

for model_name in MODELS:
    print(f"\nTesting: {model_name}")
    emb_fn = OllamaEmbeddings(model=model_name)

    dim = len(emb_fn.embed_query("test"))
    print(f"  Dimension: {dim}")

    for label, a, b, expected in TEST_PAIRS:
        try:
            sim = cosine_sim(emb_fn.embed_query(a), emb_fn.embed_query(b))
            results.append({"model": model_name, "pair": label, "expected": expected, "similarity": round(sim, 4)})
            print(f"  [{expected:6}] {label}: {sim:.4f}")
        except Exception as e:
            print(f"  ERROR on {label}: {e}")

df_sims = pd.DataFrame(results)
display(df_sims.pivot(index="pair", columns="model", values="similarity"))

## 4. Encoding Latency Benchmark

Time each model on N random sentences.

In [ ]:
BENCH_TEXTS = [
    "Alphawave is a digital transformation company.",
    "What services does Alphawave offer?",
    "AI-powered tools for enterprise clients.",
    "Contact us at info@alphawave.hr.",
    "Our team specializes in machine learning and data engineering.",
]

bench_rows = []

for model_name in MODELS:
    emb_fn = OllamaEmbeddings(model=model_name)
    t0 = time.time()
    for text in BENCH_TEXTS:
        emb_fn.embed_query(text)
    elapsed_ms = (time.time() - t0) * 1000
    avg_ms = elapsed_ms / len(BENCH_TEXTS)
    bench_rows.append({"model": model_name, "total_ms": round(elapsed_ms, 1), "avg_ms": round(avg_ms, 1)})
    print(f"{model_name}: {avg_ms:.1f} ms/text")

pd.DataFrame(bench_rows)

## 5. Similarity Heatmap

In [ ]:
if not df_sims.empty and len(MODELS) > 1:
    pivot = df_sims.pivot(index="pair", columns="model", values="similarity")
    fig, ax = plt.subplots(figsize=(max(6, len(MODELS) * 2), 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    plt.colorbar(im, ax=ax, label="cosine similarity")
    ax.set_title("Cosine Similarity by Model and Pair")
    plt.tight_layout()
    plt.show()
else:
    print("Add more models to MODELS to see a comparison heatmap.")